# TV4 — Topic baseline v1.0
Four canonical topics, three fixed TF-IDF classifiers. Select by validation Macro F1, accuracy, then fixed algorithm order. Never refit on validation/test or tune from test scores.

Local Data remains pending_leader_review. User requested completing TV4 on 2026-10-29; this notebook does not claim license approval, a shared dependency lock, or Backend acceptance.

Run with the same Python 3.13 environment as training. Existing artifacts are verified and reused; to retrain independently, use the CLI with fresh output/report/figure directories documented in docs/topic/experiment.md.


In [1]:
from pathlib import Path
import json, os, subprocess, sys
ROOT = Path.cwd()
if not (ROOT / 'ml').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / '.local-tools/topic-matplotlib'))
def run(*args):
    result = subprocess.run([sys.executable, *args], cwd=ROOT, check=True, capture_output=True,
                            text=True, encoding='utf-8', env={**os.environ, 'PYTHONIOENCODING':'utf-8'})
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
manifest = json.loads((ROOT / 'reports/data/manifest.json').read_text(encoding='utf-8'))
print('Data review status:', manifest['source']['approval_status'])
print('Source row counts:', {k:v['rows'] for k,v in manifest['splits'].items()})


Data review status: pending_leader_review
Source row counts: {'train': 11426, 'validation': 1583, 'test': 3166}


In [2]:
artifact = ROOT / 'models/topic_pipeline.joblib'
if not artifact.exists():
    run('-m', 'ml.train_topic', '--data-dir', 'data/raw', '--output-dir', 'models',
        '--report-dir', 'reports/metrics/topic', '--figure-dir', 'reports/figures/topic')
else:
    print('Reusing frozen winner; no fit or model reselection.')
run('docs/topic/complete_experiment.py')


Reusing frozen winner; no fit or model reselection.
{"algorithm": "linear_svm", "test": {"accuracy": 0.8616550852811118, "macro_precision": 0.7802037491130546, "macro_recall": 0.7138685486987416, "macro_f1": 0.7353817115066393, "per_class": [{"label": "lecturer", "precision": 0.9095925297113752, "recall": 0.9358078602620087, "f1": 0.9225139905294877, "support": 2290}, {"label": "curriculum", "precision": 0.6911262798634812, "recall": 0.708041958041958, "f1": 0.6994818652849741, "support": 572}, {"label": "facility", "precision": 0.9225352112676056, "recall": 0.903448275862069, "f1": 0.9128919860627178, "support": 145}, {"label": "other", "precision": 0.5975609756097561, "recall": 0.3081761006289308, "f1": 0.4066390041493776, "support": 159}], "confusion_matrix": [[2143, 123, 5, 19], [151, 405, 3, 13], [3, 10, 131, 1], [59, 48, 3, 49]], "label_order": ["lecturer", "curriculum", "facility", "other"], "n_rows": 3166}, "errors": 438, "reload_equal": true}



In [3]:
comparison = json.loads((ROOT / 'reports/metrics/topic/comparison.json').read_text(encoding='utf-8'))
print('Selected on validation:', comparison['selection']['algorithm'])
print('split | model | accuracy | macro precision | macro recall | macro F1')
for split in ('validation', 'test'):
    for model, scores in comparison[split].items():
        print(split, model, *(round(scores[k], 6) for k in ('accuracy','macro_precision','macro_recall','macro_f1')), sep=' | ')
winner = comparison['selection']['algorithm']
print('Per-class test metrics:', json.dumps(comparison['test'][winner]['per_class'], indent=2))
print('Confusion matrix (rows=true, columns=predicted):', comparison['test'][winner]['confusion_matrix'])
print('Figure: reports/figures/topic/confusion_matrix.png')


Selected on validation: linear_svm
split | model | accuracy | macro precision | macro recall | macro F1
validation | nb | 0.818067 | 0.626486 | 0.481143 | 0.522547
validation | logistic_regression | 0.864814 | 0.826921 | 0.635575 | 0.682239
validation | linear_svm | 0.879975 | 0.810247 | 0.737291 | 0.754173
test | nb | 0.819646 | 0.886575 | 0.500193 | 0.545789
test | logistic_regression | 0.85976 | 0.810086 | 0.65839 | 0.69819
test | linear_svm | 0.861655 | 0.780204 | 0.713869 | 0.735382
Per-class test metrics: [
  {
    "label": "lecturer",
    "precision": 0.9095925297113752,
    "recall": 0.9358078602620087,
    "f1": 0.9225139905294877,
    "support": 2290
  },
  {
    "label": "curriculum",
    "precision": 0.6911262798634812,
    "recall": 0.708041958041958,
    "f1": 0.6994818652849741,
    "support": 572
  },
  {
    "label": "facility",
    "precision": 0.9225352112676056,
    "recall": 0.903448275862069,
    "f1": 0.9128919860627178,
    "support": 145
  },
  {
    "label": "

In [4]:
errors = json.loads((ROOT / 'reports/metrics/topic/error_analysis.json').read_text(encoding='utf-8'))
reload = json.loads((ROOT / 'reports/evidence/topic/reload.json').read_text(encoding='utf-8'))
print('Errors:', errors['errors'], '/', errors['n_rows'])
print('Confusion pairs:', errors['confusions'])
print('Examples by ID (source text kept local):', errors['examples_by_id'])
print('Reload equal:', reload['before'] == reload['after_new_process'])
for text, label in zip(reload['samples'], reload['before']):
    print(text, '=>', label)


Errors: 438 / 3166
Confusion pairs: {'curriculum -> lecturer': 151, 'lecturer -> curriculum': 123, 'other -> lecturer': 59, 'other -> curriculum': 48, 'lecturer -> other': 19, 'curriculum -> other': 13, 'facility -> curriculum': 10, 'lecturer -> facility': 5, 'other -> facility': 3, 'facility -> lecturer': 3, 'curriculum -> facility': 3, 'facility -> other': 1}
Examples by ID (source text kept local): [{'id': 'test:000013', 'actual': 'other', 'predicted': 'facility'}, {'id': 'test:000014', 'actual': 'curriculum', 'predicted': 'lecturer'}, {'id': 'test:000019', 'actual': 'curriculum', 'predicted': 'lecturer'}, {'id': 'test:000024', 'actual': 'lecturer', 'predicted': 'curriculum'}, {'id': 'test:000041', 'actual': 'lecturer', 'predicted': 'facility'}, {'id': 'test:000044', 'actual': 'curriculum', 'predicted': 'lecturer'}, {'id': 'test:000049', 'actual': 'curriculum', 'predicted': 'lecturer'}, {'id': 'test:000050', 'actual': 'lecturer', 'predicted': 'curriculum'}, {'id': 'test:000051', 'ac

## Interpretation and handoff
See docs/topic/report.md for measured minority-class behavior and manually inspected multi-topic cases. A single-label model cannot represent every aspect of a multi-topic sentence. The authored examples above are inference demonstrations, not labeled accuracy measurements.

Metadata binds the winner to exact packages, preprocessing, and dataset manifest. Both real pipelines must be tested in the Backend environment before integration acceptance. Corpus/model redistribution and leader review remain pending.
